In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType,FloatType
import pyspark.sql.functions as F

In [0]:
catalog_name = "ecommerce"

# Define schema for the data file 
brand_schema = StructType([
    StructField("brand_code",StringType(),False),
    StructField("brand_name",StringType(),True),
    StructField("category_code",StringType(),True)
])

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw/brands/*.csv"
df = spark.read.option("header","true").option("delimiter",",").schema(brand_schema).csv(raw_data_path)

# only syntax change
# df = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/ecommerce/source_data/raw/brands/brands.csv")

# Add MetaData Columns
df = df.withColumn("_source_file", F.col("_metadata.file_path"))\
    .withColumn("ingested_at",F.current_timestamp())

df.display()


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

# CATEGORY.csv

In [0]:
category_schema = StructType([
    StructField("category_code",StringType(),False),
    StructField("category_name",StringType(),True)
])

# Load daat using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/category/*.csv"

df_raw = spark.read.option("header","true").option("delimiter",",").schema(category_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at",F.current_timestamp())\
            .withColumn("_source_file",F.col("_metadata.file_path"))

# write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_category)
df_raw.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_category")



# PRODUCTS.csv

In [0]:
product_schema =StructType([
    StructField("product_id",StringType(),False),
    StructField("sku",StringType(),True),
    StructField("category_code",StringType(),True),
    StructField("brand_code",StringType(),True),
    StructField("color",StringType(),True),
    StructField("size",StringType(),True),
    StructField("material",StringType(),True),
    StructField("weight_grams",StringType(),True),  
    StructField("length_cm",StringType(),True),
    StructField("width_cm",FloatType(),True),
    StructField("height_cm",FloatType(),True),
    StructField("rating_count",IntegerType(),True),
    StructField("file_name",StringType(),False), 
    StructField("ingest_timestamp",TimestampType(),False),
])

# Load Data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"

df = spark.read.option("header","true").option("delimiter",",").schema(product_schema).csv(raw_data_path)

# Add metadata columns
df = df.withColumn("file_name",F.col("_metadata.file_name"))\
            .withColumn("_ingested_timestamp",F.current_timestamp())

# write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_products")

In [0]:
display(df)